In [1]:
import sys
import os
import importlib

project_root = "/Users/xinghebai/Desktop/Master Project"
habitat_lab_path = os.path.join(project_root, "habitat-lab", "habitat")

# Clean up old imports
if 'habitat' in sys.modules:
    del sys.modules['habitat']

# Add correct path
if habitat_lab_path not in sys.path:
    sys.path.insert(0, habitat_lab_path)
    print(f"Added path: {habitat_lab_path}")

# Verify source existence before import
init_file = os.path.join(habitat_lab_path, "habitat", "__init__.py")
if os.path.exists(init_file):
    print(f"Found source code at: {init_file}")
else:
    print(f"Warning: Still cannot find __init__.py at {init_file}")

# Import
import habitat
importlib.reload(habitat)

try:
    from habitat.core.env import Env
    print("SUCCESS: habitat.core.env imported!")
except ImportError as e:
    print(f"Import Error: {e}")


Added path: /Users/xinghebai/Desktop/Master Project/habitat-lab/habitat
Found source code at: /Users/xinghebai/Desktop/Master Project/habitat-lab/habitat/habitat/__init__.py


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


SUCCESS: habitat.core.env imported!


PluginManager::Manager: duplicate static plugin StbImageImporter, ignoring
PluginManager::Manager: duplicate static plugin GltfImporter, ignoring
PluginManager::Manager: duplicate static plugin BasisImporter, ignoring
PluginManager::Manager: duplicate static plugin AssimpImporter, ignoring
PluginManager::Manager: duplicate static plugin AnySceneImporter, ignoring
PluginManager::Manager: duplicate static plugin AnyImageImporter, ignoring


In [2]:
import torch
import torch.optim as optim
import numpy as np
from collections import deque
import os

# Import the classes we created
from PolicyAgent import PolicyAgent 
from EnvAdapter import HabitatEnvAdapter
from TeacherScorer import TeacherScorer

# Hyperparameters for Baseline AutoPhoto reproduction
LR = 1e-4
GAMMA = 0.99
UPDATE_TIMESTEP = 2000  
MAX_EPISODES = 1000
MAX_STEPS_PER_EPISODE = 500

def train():
    # Initialize Environment and Agent
    config_path = "/Users/xinghebai/Desktop/Master Project/habitat-lab/habitat/habitat/config/benchmark/nav/pointnav/pointnav_gibson.yaml" 
    
    if not os.path.exists(config_path):
        print(f"Error: Config not found at {config_path}. Please check your path.")
        return

    print(f"Initializing Habitat Environment with {config_path}...")
    

    scorer = TeacherScorer(mode="deep", weights_path="siglip_head_best.pth")
    
    # Pass the scorer to the environment
    env = HabitatEnvAdapter(config_path=config_path, scorer_model=scorer)
    
    agent = PolicyAgent() 
    optimizer = optim.Adam(agent.policy.parameters(), lr=LR)
    
    memory = [] 
    time_step = 0
    
    print("Starting SmartShot Training with Teacher Scorer...")
    
    for i_episode in range(1, MAX_EPISODES + 1):
        state = env.reset()
        agent.reset() # Reset LSTM hidden state for new episode
        episode_reward = 0
        
        for t in range(MAX_STEPS_PER_EPISODE):
            time_step += 1
            
            # Select Action
            action_idx, confidence = agent.act(
                state['frame'], 
                state['score'].item(), 
                state['score'].item() - state['delta'].item(), 
                state['egomotion']
            )
            
            # Execute Step in Habitat
            next_state, reward, done, info = env.step(action_idx)
            
            # Store transition for PPO update
            memory.append((reward, done))
            
            state = next_state
            episode_reward += reward
            
            # Simple Update Logic 
            if time_step % UPDATE_TIMESTEP == 0:
                print(f"--> Time Step {time_step}: Updating Policy...")
                update_policy_simple(agent, optimizer, memory)
                memory = [] 
                
            if done:
                break
                
        print(f"Episode {i_episode}\t Total Reward: {episode_reward:.2f}\t Steps: {t}")

def update_policy_simple(agent, optimizer, memory):
    """
    A simplified update step to verify the gradient loop works.
    """
    optimizer.zero_grad()
    
    # Dummy loss to verify backprop works 
    loss = torch.tensor(0.0, requires_grad=True) 
    for r, _ in memory:
        loss = loss + r
        
    # We want to maximize reward, so minimize negative reward
    loss = -loss / (len(memory) + 1e-8)
    loss.backward()
    optimizer.step()

if __name__ == "__main__":
    train()

Initializing Habitat Environment with /Users/xinghebai/Desktop/Master Project/habitat-lab/habitat/habitat/config/benchmark/nav/pointnav/pointnav_gibson.yaml...
Initializing TeacherScorer (Mode: deep) on mps...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
2025-12-05 21:39:02,054 Initializing dataset PointNav-v1


✅ Successfully loaded SigLIP weights from siglip_head_best.pth


2025-12-05 21:41:26,282 initializing sim Sim-v0
PluginManager::Manager: duplicate static plugin2025-12-05 21:41:28,931 Initializing task Nav-v0
 StbImageImporter, ignoring
PluginManager::Manager: duplicate static plugin GltfImporter, ignoring
PluginManager::Manager: duplicate static plugin BasisImporter, ignoring
PluginManager::Manager: duplicate static plugin AssimpImporter, ignoring
PluginManager::Manager: duplicate static plugin AnySceneImporter, ignoring
PluginManager::Manager: duplicate static plugin AnyImageImporter, ignoring
[15:47:03:579350]:[Warning]:[Metadata] SceneDatasetAttributes.cpp(107)::addNewSceneInstanceToDataset : Dataset : 'default' : Lighting Layout Attributes 'no_lights' specified in Scene Attributes but does not exist in dataset, so creating default.
[15:47:03:580446]:[Warning]:[Scene] SemanticScene.h(331)::checkFileExists : ::loadSemanticSceneDescriptor: File `/Users/xinghebai/Desktop/Master Project/habitat-lab/data/scene_datasets/gibson/Roeville.scn` does not e

Renderer: Apple M2 Pro by Apple
OpenGL version: 4.1 Metal - 89.4
Using optional features:
    GL_ARB_vertex_array_object
    GL_ARB_ES2_compatibility
    GL_ARB_separate_shader_objects
    GL_ARB_texture_storage
    GL_EXT_texture_filter_anisotropic
    GL_EXT_debug_label
    GL_EXT_debug_marker
Using driver workarounds:
    no-layout-qualifiers-on-old-glsl
    apple-buffer-texture-unbind-on-buffer-modify
Starting SmartShot Training with Teacher Scorer...
Episode 1	 Total Reward: 1.10	 Steps: 2
Episode 2	 Total Reward: -0.26	 Steps: 1
Episode 3	 Total Reward: 2.69	 Steps: 3
Episode 4	 Total Reward: 2.84	 Steps: 3
Episode 5	 Total Reward: -0.11	 Steps: 0
Episode 6	 Total Reward: -1.33	 Steps: 2
Episode 7	 Total Reward: -1.06	 Steps: 1
Episode 8	 Total Reward: -1.74	 Steps: 10
Episode 9	 Total Reward: -0.12	 Steps: 2
Episode 10	 Total Reward: -0.11	 Steps: 0
Episode 11	 Total Reward: -0.11	 Steps: 0
Episode 12	 Total Reward: -0.11	 Steps: 0
Episode 13	 Total Reward: -0.83	 Steps: 3
Episo